In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

sys.path.append("..")  # allows importing from project root
from data_solar import (
    load_solar_generation,
    normalize_by_max,
    scale_to_pv_capacity,
    prepare_solar_series,
    validate_solar_series,
    save_solar,
)

# ── Config ──────────────────────────────────────────────────────
SOLAR_PATH  = r"C:\Users\vp532\OneDrive\Desktop\Mini_project-2\datasets\Solar Power Generation Data\Plant_1_Generation_Data.csv"
OUTPUT_PATH = r"C:\Users\vp532\OneDrive\Desktop\Mini_project-2\datasets\processed\solar_scaled.csv"
N_DAYS      = 30          # dataset covers ~34 days — keep ≤ 34
PV_CAPACITY = 3.0         # MW — assumed installed PV capacity

print("Imports OK")

: 

---
## Inspect Raw CSV
Understand structure before any processing.

In [ ]:
raw_df = pd.read_csv(SOLAR_PATH)

print(f"Shape             : {raw_df.shape}")
print(f"Columns           : {raw_df.columns.tolist()}")
print(f"Unique timestamps : {raw_df['DATE_TIME'].nunique()}")
print(f"Unique inverters  : {raw_df['SOURCE_KEY'].nunique()}")
print(f"\nDC_POWER stats (per inverter per timestamp):")
print(raw_df['DC_POWER'].describe().to_string())
print(f"\nFirst 3 rows:")
raw_df.head(3)

---
## Load and Aggregate
Sum DC_POWER across all 22 inverters per timestamp → total plant output in MW.

In [ ]:
hourly_mw = load_solar_generation(SOLAR_PATH)

print(f"Shape          : {hourly_mw.shape[0]} hours  ({hourly_mw.shape[0] / 24:.1f} days)")
print(f"Date range     : {hourly_mw.index[0]}  →  {hourly_mw.index[-1]}")
print(f"Max generation : {hourly_mw.max():.6f} MW  ({hourly_mw.max()*1000:.2f} kW)")
print(f"Mean (all hrs) : {hourly_mw.mean():.6f} MW")
print(f"Night hours    : {(hourly_mw == 0).sum()}  ({(hourly_mw == 0).sum()/len(hourly_mw)*100:.1f}%)")
print(f"NaN count      : {hourly_mw.isna().sum()}")

### Raw Solar — First 7 Days

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
hourly_mw.iloc[:168].plot(ax=ax, color="darkorange", linewidth=1)
ax.set_title("Raw Total Plant DC Power — First 7 Days (22 Inverters Summed)", fontsize=13)
ax.set_xlabel("Datetime")
ax.set_ylabel("DC Power (MW)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
##  Normalize → Scale → Trim

In [ ]:
# Step 1: Trim to simulation horizon first
trimmed = hourly_mw.iloc[: N_DAYS * 24]
print(f"After trim       →  {len(trimmed)} hours  ({N_DAYS} days)")

# Step 2: Max-normalize to [0, 1]
solar_norm = normalize_by_max(trimmed)
print(f"After normalize  →  Max: {solar_norm.max():.4f}  |  Min: {solar_norm.min():.4f}")

# Step 3: Scale to PV capacity
solar_scaled = scale_to_pv_capacity(solar_norm, pv_capacity_mw=PV_CAPACITY)
print(f"After scaling    →  Max: {solar_scaled.max():.4f} MW  |  Min: {solar_scaled.min():.4f} MW")

# Reset index for simulation loop
solar_scaled = solar_scaled.reset_index(drop=True)

---
##  Validate

In [ ]:
assert solar_scaled.isna().sum() == 0,          "FAIL: NaN values found"
assert solar_scaled.min() >= 0.0,               "FAIL: Negative values found"
assert solar_scaled.max() <= PV_CAPACITY + 1e-6, "FAIL: Exceeds PV capacity"

peak_idx  = solar_scaled.idxmax()
peak_day  = peak_idx // 24
peak_hour = peak_idx % 24

night_hours  = (solar_scaled == 0.0).sum()
active_hours = (solar_scaled > 0.0).sum()

print("All validation checks passed.")
print(f"\nFinal series summary:")
print(f"  Length       : {len(solar_scaled)} hours")
print(f"  Min          : {solar_scaled.min():.6f} MW")
print(f"  Max          : {solar_scaled.max():.4f} MW  (Day {peak_day}, Hour {peak_hour}:00)")
print(f"  Mean (all)   : {solar_scaled.mean():.4f} MW")
print(f"  Mean (active): {solar_scaled[solar_scaled > 0].mean():.4f} MW")
print(f"  Night hours  : {night_hours}  ({night_hours/len(solar_scaled)*100:.1f}%)")
print(f"  Active hours : {active_hours}  ({active_hours/len(solar_scaled)*100:.1f}%)")

---
## Visualize Final Solar Profile

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7))

# ── Full simulation horizon ──────────────────────────────────────
axes[0].plot(solar_scaled.values, color="darkorange", linewidth=0.8)
axes[0].axhline(solar_scaled.mean(), color="steelblue", linestyle="--",
                linewidth=1.2, label=f"Mean (all hrs): {solar_scaled.mean():.3f} MW")
axes[0].axhline(PV_CAPACITY, color="red", linestyle=":",
                linewidth=1.2, label=f"PV capacity: {PV_CAPACITY} MW")
axes[0].set_title(f"Final Solar Profile — {N_DAYS}-Day Simulation Horizon", fontsize=13)
axes[0].set_xlabel("Timestep (hours)")
axes[0].set_ylabel("p_solar (MW)")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# ── First 7 days zoomed ──────────────────────────────────────────
axes[1].fill_between(range(168), solar_scaled.values[:168],
                     alpha=0.4, color="darkorange")
axes[1].plot(solar_scaled.values[:168], color="darkorange", linewidth=1.2)
axes[1].set_title("Zoomed — First 7 Days (Day/Night Cycle)", fontsize=13)
axes[1].set_xlabel("Timestep (hours)")
axes[1].set_ylabel("p_solar (MW)")
axes[1].xaxis.set_major_locator(ticker.MultipleLocator(24))
axes[1].xaxis.set_minor_locator(ticker.MultipleLocator(6))
axes[1].grid(True, which="major", alpha=0.4)
axes[1].grid(True, which="minor", alpha=0.15)

plt.tight_layout()
plt.show()

---
## Save

In [ ]:
solar_scaled.to_csv(OUTPUT_PATH, index=False, header=['p_solar_mw'])
print(f"Saved → {OUTPUT_PATH}")
print(f"Shape : {len(solar_scaled)} rows x 1 column (p_solar_mw)")

# Quick read-back check
check = pd.read_csv(OUTPUT_PATH)
print(f"\nRead-back check:")
print(check.head())

In [ ]:
import pandas as pd
df = pd.read_csv(r'C:\Users\vp532\OneDrive\Desktop\Mini_project-2\datasets\Solar Power Generation Data\Plant_1_Generation_Data.csv')
print(df.columns.tolist())
print(df.dtypes)
print(df.head(3))
print(df.shape)
print(df.describe())

In [ ]:
print(df['SOURCE_KEY'].nunique())           # how many inverters
print(df['DATE_TIME'].nunique())            # how many unique timestamps
print(df['DATE_TIME'].iloc[0])              # confirm format
print(df['DATE_TIME'].iloc[1])              # check if 15-min spacing